<a href="https://colab.research.google.com/github/ADRAKECROWDER/AIML2003-nlp/blob/main/Week6/ItineraryBuilder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Cell 1: Setup - install packages, load API keys, configure Gemini, and test the connection

# Install required packages
!pip install -q google-genai requests pandas

# Import Gemini SDK
from google import genai
from google.genai import types

# Import Colab Secrets
from google.colab import userdata

# Import display/data tools
from IPython.display import display, HTML
import pandas as pd

# Import standard tools
import requests
import json
import datetime
import math
import time
import re

# LLM
MODEL_ID = "gemini-flash-latest"

# Load keys
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GEOAPIFY_API_KEY = userdata.get("GEOAPIFY_API_KEY")
FOURSQUARE_API_KEY = userdata.get("FOURSQUARE_API_KEY")

# Check required keys
missing_keys = []

if not GEMINI_API_KEY:
    missing_keys.append("GEMINI_API_KEY")

if not GEOAPIFY_API_KEY:
    missing_keys.append("GEOAPIFY_API_KEY")

if not FOURSQUARE_API_KEY:
    missing_keys.append("FOURSQUARE_API_KEY")

if missing_keys:
    raise ValueError(f"Missing Colab Secrets: {missing_keys}")

# Create Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Test Gemini connection
test_response = client.models.generate_content(
    model=MODEL_ID,
    contents="Reply with one sentence confirming the Gemini API connection works."
)

# Print connection test
print(test_response.text)

The Gemini API connection is successful and working correctly.


###Cell 2

### Cell 2 Tool List

| Tool | Where it is used | Why it is used |
|---|---|---|
| `calculator()` | Cell 9 agent routing test | Tests the lab’s required math/tool-calling behavior using travel budget math, such as a hotel subtotal. |
| `get_datetime()` | Cell 9 agent routing test | Tests the lab’s required date/time tool behavior. |
| `geocode_destination()` | Cell 2 inside `recommend_destinations()` and `search_foursquare_places()` | Uses Geoapify to turn a destination name into latitude and longitude so destinations can be validated and Foursquare can search by radius. |
| `recommend_destinations()` | Cell 5 and Cell 9 | Uses Gemini to generate 10 destination recommendations from the user profile, then Geoapify validates those destinations. |
| `search_foursquare_places()` | Cell 2 inside `search_places()` and `search_activity_options()` | Calls Foursquare directly to find hotels, restaurants, spas, yoga studios, beaches, trails, and other local places. |
| `search_places()` | Cell 6 and Cell 9 | Searches for hotels or other place categories by translating general labels like `hotels` into Foursquare search terms like `hotel`. |
| `search_activity_options()` | Cell 7 and Cell 9 | Pulls 5 specific activity options for each selected activity category. It uses Foursquare first, then Gemini fallback if fewer than 5 results come back. |
| `generate_llm_activity_fallbacks()` | Cell 2 inside `search_activity_options()` | Helper function. Fills missing activity options when Foursquare does not return enough results. |
| `add_required_beach_option_if_needed()` | Cell 2 inside `search_activity_options()` | Helper function. Adds Waikiki Beach only when the selected destination is Hawaii/Oahu/Waikiki and the user selected beach time. |
| `estimate_budget_fit()` | Cell 5 and Cell 9 | Uses Gemini to judge whether each destination fits the user’s budget using general travel-cost reasoning. |
| `create_user_profile()` | Cell 5 and Cell 9 | Validates that the user profile includes the required fields before the travel workflow continues. |
| `build_itinerary()` | Cell 8 and Cell 9 | Builds the final day-by-day itinerary and checks for invalid day or time assignments. |

In [5]:
# Cell 2: Tool Functions - calculator, date/time, destination recommendation, Foursquare search, Geoapify geocoding, budget fit, profile, and itinerary helpers

# Calculator tool for travel budget math
def calculator(expression: str) -> str:
    safe_namespace = {
        "math": math,
        "sqrt": math.sqrt,
        "pow": pow,
        "abs": abs,
        "round": round,
        "min": min,
        "max": max
    }

    try:
        result = eval(expression, {"__builtins__": {}}, safe_namespace)
        return str(result)
    except Exception as e:
        return f"Calculator error: {type(e).__name__}: {e}"


# Date/time tool
def get_datetime() -> str:
    now = datetime.datetime.now()
    return now.strftime("%A, %B %d, %Y, %I:%M %p local time")


# Helper to parse comma-separated numbers like 1,3,5
def parse_number_list(text):
    numbers = []

    for part in text.split(","):
        part = part.strip()

        if part.isdigit():
            numbers.append(int(part))

    return numbers


# Search radius used for local place searches
LOCAL_SEARCH_RADIUS_METERS = 70000


# Geoapify destination lookup
def geocode_destination(destination: str) -> str:
    url = "https://api.geoapify.com/v1/geocode/search"

    params = {
        "text": destination,
        "format": "json",
        "limit": 1,
        "apiKey": GEOAPIFY_API_KEY
    }

    response = requests.get(url, params=params, timeout=20)

    if response.status_code != 200:
        raise RuntimeError(f"Geoapify geocoding failed: {response.status_code} {response.text}")

    data = response.json()

    if not data.get("results"):
        raise ValueError(f"No destination found for: {destination}")

    result = data["results"][0]

    output = {
        "searched_destination": destination,
        "formatted_destination": result.get("formatted"),
        "city": result.get("city"),
        "state": result.get("state"),
        "country": result.get("country"),
        "lat": result.get("lat"),
        "lon": result.get("lon")
    }

    return json.dumps(output, indent=2)


# Recommend 10 destinations from the profile, then validate them with Geoapify
def recommend_destinations(profile_json: str, required_destination: str = "Waikiki, Honolulu, Hawaii", total_options: int = 10) -> str:
    profile = json.loads(profile_json)

    prompt = f"""
You are a travel destination recommendation agent.

Recommend travel destinations based on the user profile.
Do not use a fixed list.
Use the user's origin, budget, trip length, destination interests, hotel style, pace, and crowd tolerance.

The final list must include this required project destination:
{required_destination}

Return JSON only.
Return a list of exactly {total_options} destination strings.
Each destination should be specific enough for geocoding, such as:
"Waikiki, Honolulu, Hawaii"

Do not include explanations.
Do not include duplicate destinations.

User profile:
{profile_json}
"""

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )

    text = response.text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        raw_destinations = json.loads(text)
    except Exception:
        raise ValueError(f"Gemini destination output was not valid JSON: {text}")

    if not isinstance(raw_destinations, list):
        raise ValueError("Gemini destination output must be a JSON list.")

    # Clean duplicates
    cleaned_destinations = []

    for destination in raw_destinations:
        destination_text = str(destination).strip()

        if destination_text and destination_text.lower() not in [d.lower() for d in cleaned_destinations]:
            cleaned_destinations.append(destination_text)

    # Require Waikiki for later CV handoff
    if required_destination.lower() not in [d.lower() for d in cleaned_destinations]:
        cleaned_destinations.insert(0, required_destination)

    # Validate destinations with Geoapify
    validated_destinations = []

    for destination in cleaned_destinations:
        try:
            geo_json = geocode_destination(destination)
            geo_data = json.loads(geo_json)

            validated_destinations.append({
                "destination": geo_data.get("formatted_destination") or destination,
                "original_recommendation": destination,
                "city": geo_data.get("city"),
                "state": geo_data.get("state"),
                "country": geo_data.get("country"),
                "lat": geo_data.get("lat"),
                "lon": geo_data.get("lon"),
                "validation_source": "Geoapify geocoding"
            })

        except Exception:
            continue

        if len(validated_destinations) == total_options:
            break

    if len(validated_destinations) < total_options:
        raise ValueError(
            f"Only {len(validated_destinations)} destinations were validated. "
            "Rerun the cell so Gemini can generate a cleaner list."
        )

    output = {
        "profile_used": profile,
        "required_destination": required_destination,
        "destination_count": len(validated_destinations),
        "destinations": validated_destinations
    }

    return json.dumps(output, indent=2)


# Foursquare place search tool for hotels, restaurants, spas, yoga, beaches, hiking, and attractions
def search_foursquare_places(destination: str, query: str, limit: int = 5, radius_meters: int = LOCAL_SEARCH_RADIUS_METERS) -> str:
    # Geocode the destination first so Foursquare can use ll + radius
    geo_data = json.loads(geocode_destination(destination))

    lat = geo_data["lat"]
    lon = geo_data["lon"]

    # Foursquare Places API endpoint
    url = "https://places-api.foursquare.com/places/search"

    # Foursquare request headers
    headers = {
        "Authorization": f"Bearer {FOURSQUARE_API_KEY}",
        "accept": "application/json",
        "X-Places-Api-Version": "2025-02-05"
    }

    # Use ll + radius, not near + radius
    params = {
        "ll": f"{lat},{lon}",
        "query": query,
        "radius": radius_meters,
        "limit": limit,
        "fields": "fsq_place_id,name,location,categories,distance,website"
    }

    # Call Foursquare
    response = requests.get(url, headers=headers, params=params, timeout=20)

    # Stop if the API call fails
    if response.status_code != 200:
        raise RuntimeError(f"Foursquare search failed: {response.status_code} {response.text}")

    # Parse results
    data = response.json()
    raw_results = data.get("results", [])
    places = []

    # Clean each place result
    for item in raw_results:
        location = item.get("location", {})
        categories = item.get("categories", [])

        category_names = [
            category.get("name")
            for category in categories
            if category.get("name")
        ]

        places.append({
            "name": item.get("name", "Unnamed place"),
            "address": location.get("formatted_address") or ", ".join(location.get("address_lines", [])),
            "city": location.get("locality"),
            "state": location.get("region"),
            "country": location.get("country"),
            "postcode": location.get("postcode"),
            "categories": category_names,
            "distance_meters": item.get("distance"),
            "website": item.get("website"),
            "fsq_place_id": item.get("fsq_place_id"),
            "result_source": "Foursquare API",
            "source": "Foursquare API"
        })

    # Return consistent JSON output
    output = {
        "query": query,
        "destination": destination,
        "lat": lat,
        "lon": lon,
        "radius_meters": radius_meters,
        "results_count": len(places),
        "places": places
    }

    return json.dumps(output, indent=2)

# Geoapify text search kept only as a backup utility, not used for activity search
def search_geoapify_text(query: str, destination: str, limit: int = 5, radius_meters: int = LOCAL_SEARCH_RADIUS_METERS) -> str:
    geo_data = json.loads(geocode_destination(destination))

    lat = geo_data["lat"]
    lon = geo_data["lon"]

    url = "https://api.geoapify.com/v1/geocode/search"

    params = {
        "text": query,
        "format": "json",
        "limit": limit,
        "filter": f"circle:{lon},{lat},{radius_meters}",
        "bias": f"proximity:{lon},{lat}",
        "apiKey": GEOAPIFY_API_KEY
    }

    response = requests.get(url, params=params, timeout=20)

    if response.status_code != 200:
        raise RuntimeError(f"Geoapify text search failed: {response.status_code} {response.text}")

    data = response.json()
    places = []

    for item in data.get("results", []):
        places.append({
            "name": item.get("name") or item.get("formatted"),
            "address": item.get("formatted"),
            "city": item.get("city"),
            "state": item.get("state"),
            "country": item.get("country"),
            "lat": item.get("lat"),
            "lon": item.get("lon"),
            "website": item.get("website"),
            "result_source": "Geoapify API",
            "source": "Geoapify API"
        })

    output = {
        "query": query,
        "destination": destination,
        "radius_meters": radius_meters,
        "results_count": len(places),
        "places": places
    }

    return json.dumps(output, indent=2)


# Wrapper so the notebook can call search_places while using Foursquare underneath
def search_places(destination: str, category: str, limit: int = 5, radius_meters: int = LOCAL_SEARCH_RADIUS_METERS) -> str:
    category_to_query = {
        "hotels": "hotel",
        "hotel": "hotel",
        "accommodation": "hotel",
        "accommodation.hotel": "hotel",
        "resorts": "resort",
        "resort": "resort",

        "restaurants": "restaurant",
        "restaurant": "restaurant",
        "food": "restaurant",
        "healthy dining": "healthy restaurant",
        "fine dining": "fine dining restaurant",
        "beachfront dining": "beachfront restaurant",

        "spa": "spa",
        "spas": "spa",
        "spa treatment": "spa",
        "spa day": "spa",
        "wellness": "wellness spa",

        "yoga": "yoga studio",
        "yoga class": "yoga studio",

        "beach": "beach",
        "beaches": "beach",
        "beach time": "beach",

        "hiking": "hiking trail",
        "hiking trail": "hiking trail",
        "trails": "hiking trail",

        "parks": "park",
        "park": "park",

        "shopping": "shopping",
        "museums": "museum",
        "museum": "museum",
        "nightlife": "nightlife",
        "bars": "bar",
        "clubs": "nightclub",
        "attractions": "tourist attraction",
        "activities": "tourist attraction"
    }

    query = category_to_query.get(category.lower().strip(), category)

    return search_foursquare_places(
        destination=destination,
        query=query,
        limit=limit,
        radius_meters=radius_meters
    )


# Activity search term cleanup layer, not recommendation data
activity_search_map = {
    "spa day": "spa",
    "spa treatment": "spa",
    "massage": "massage spa",
    "couples spa": "spa",
    "yoga class": "yoga studio",
    "meditation": "meditation center",

    "beach time": "beach",
    "beach": "beach",
    "snorkeling": "snorkeling",
    "surfing": "surfing",
    "paddleboarding": "paddleboarding",
    "sunset cruise": "sunset cruise",
    "boat tour": "boat tour",
    "private cabana": "beach club",
    "resort pool": "resort",

    "hiking": "hiking trail",
    "nature walk": "nature trail",
    "scenic overlooks": "scenic lookout",
    "wildlife viewing": "wildlife viewing",
    "botanical garden": "botanical garden",

    "beachfront dining": "beachfront restaurant",
    "local restaurants": "restaurant",
    "healthy dining": "healthy restaurant",
    "fine dining": "fine dining restaurant",
    "food tour": "food tour",
    "coffee shops": "coffee shop",
    "dessert spots": "dessert",

    "museums": "museum",
    "historic sites": "historic site",
    "local landmarks": "landmark",
    "cultural tour": "cultural tour",
    "art galleries": "art gallery",

    "shopping": "shopping",
    "boutiques": "boutique",
    "local markets": "market",
    "shopping districts": "shopping mall",
    "souvenir shops": "souvenir shop",

    "nightlife": "nightlife",
    "bars": "bar",
    "clubs": "nightclub",
    "live music": "live music",
    "rooftop bar": "rooftop bar",

    "coastal walk": "coastal walk",
    "scenic drive": "scenic drive",
    "lake activities": "lake",
    "mountain biking": "mountain biking",
    "ziplining": "ziplining",
    "stargazing": "stargazing",
    "theater": "theater",
    "walking tour": "walking tour",
    "architecture tour": "architecture tour",
    "rock climbing": "rock climbing",
    "kayaking": "kayaking",
    "ATV tour": "ATV tour",
    "skiing": "skiing",
    "snowboarding": "snowboarding",
    "hot springs": "hot springs",
    "scenic gondola": "gondola",
    "ranger-led tour": "visitor center",
    "nature photography": "nature",
    "camping": "camping",
    "visitor center": "visitor center",
    "farmers market": "market",
    "cooking class": "cooking class",
    "street food": "restaurant",
    "historic sites": "historic site",
    "heritage walk": "historic walk",
    "artisan markets": "market",
    "luxury shopping": "shopping",
    "comedy show": "comedy club",
    "late-night dining": "restaurant",
    "aquarium": "aquarium",
    "zoo": "zoo",
    "easy walking tour": "walking tour",
    "interactive museum": "museum",
    "family restaurants": "family restaurant",
    "sunset dinner": "restaurant",
    "scenic walk": "scenic walk",
    "wine tasting": "wine tasting",
    "private tour": "private tour",
    "beach picnic": "beach",
    "golf": "golf",
    "eco tour": "eco tour",
    "birdwatching": "birdwatching",
    "conservation tour": "conservation tour"
}


# Gemini fallback to fill missing activity options when Foursquare returns fewer than 5
def generate_llm_activity_fallbacks(destination: str, activity_category: str, existing_places_json: str, needed_count: int) -> list:
    prompt = f"""
You are helping fill missing travel activity suggestions for a trip-planning notebook.

The Foursquare API was tried first, but it returned fewer than 5 usable options.

Use general travel knowledge to suggest likely activities or places for the selected destination.
Do not invent exact addresses.
Do not claim these came from the API.
Label these as Gemini fallback suggestions.

Destination:
{destination}

Activity category:
{activity_category}

Existing API places:
{existing_places_json}

Number of fallback suggestions needed:
{needed_count}

Return JSON only.
Return a list of exactly {needed_count} objects.
Each object must include:
name
address
city
state
country
lat
lon
website
result_source
source

Rules:
- result_source must be "Gemini fallback"
- source must be "Gemini fallback"
- address can be "Address not provided by Gemini fallback"
- lat and lon can be null
- website can be null
"""

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )

    text = response.text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        fallback_places = json.loads(text)

        if isinstance(fallback_places, list):
            return fallback_places[:needed_count]

        return []

    except Exception:
        return []


# Add required Waikiki Beach option when beach time is selected for Oahu/Honolulu/Waikiki
def add_required_beach_option_if_needed(destination: str, category: str, places: list) -> list:
    destination_lower = destination.lower()
    category_lower = category.lower()

    is_oahu_context = (
        "waikiki" in destination_lower
        or "honolulu" in destination_lower
        or "oahu" in destination_lower
        or "hawaii" in destination_lower
    )

    is_beach_time = category_lower in ["beach time", "beach"]

    if not is_oahu_context or not is_beach_time:
        return places

    existing_names = [str(place.get("name", "")).lower() for place in places]

    if any("waikiki beach" in name for name in existing_names):
        return places

    waikiki_option = {
        "name": "Waikiki Beach",
        "address": "Waikiki, Honolulu, Hawaii",
        "city": "Honolulu",
        "state": "Hawaii",
        "country": "United States",
        "lat": None,
        "lon": None,
        "website": None,
        "result_source": "Required project handoff option",
        "source": "Required project handoff option"
    }

    return [waikiki_option] + places


# Search 5 Foursquare/fallback results for each selected activity category
def search_activity_options(destination: str, selected_activity_categories_json: str, limit_per_category: int = 5) -> str:
    selected_activity_categories = json.loads(selected_activity_categories_json)
    grouped_results = {}

    for category in selected_activity_categories:
        search_term = activity_search_map.get(category, category)

        try:
            result_json = search_foursquare_places(
                destination=destination,
                query=search_term,
                limit=limit_per_category,
                radius_meters=LOCAL_SEARCH_RADIUS_METERS
            )

            result_data = json.loads(result_json)
            places = result_data.get("places", [])

        except Exception:
            places = []

        places = add_required_beach_option_if_needed(destination, category, places)

        unique_places = []
        seen_names = set()

        for place in places:
            name = str(place.get("name", "")).strip()

            if name and name.lower() not in seen_names:
                unique_places.append(place)
                seen_names.add(name.lower())

        places = unique_places[:limit_per_category]
        needed_count = limit_per_category - len(places)

        if needed_count > 0:
            fallback_places = generate_llm_activity_fallbacks(
                destination=destination,
                activity_category=category,
                existing_places_json=json.dumps(places),
                needed_count=needed_count
            )

            places.extend(fallback_places)

        for place in places:
            place["matched_activity_category"] = category
            place["search_term_used"] = search_term

            if not place.get("result_source"):
                place["result_source"] = place.get("source", "Unknown")

            if not place.get("source"):
                place["source"] = place.get("result_source", "Unknown")

        grouped_results[category] = places[:limit_per_category]

    output = {
        "destination": destination,
        "selected_activity_categories": selected_activity_categories,
        "search_radius_meters": LOCAL_SEARCH_RADIUS_METERS,
        "primary_activity_source": "Foursquare API",
        "fallback_source": "Gemini fallback only when Foursquare returns fewer than 5",
        "results_by_category": grouped_results
    }

    return json.dumps(output, indent=2)


# Gemini budget reasoning tool for destination fit
def estimate_budget_fit(profile_json: str, destination_options_json: str) -> str:
    prompt = f"""
You are a travel budget feasibility estimator.

Use general travel-cost reasoning. Do not claim these are live prices.
Classify each destination as:
- realistic
- tight
- not recommended

Consider:
- origin
- age
- traveling with children
- number of travelers
- total trip budget
- trip length
- destination type
- likely flight distance
- island or remote location
- general lodging cost
- food cost
- activity cost
- hotel budget tier
- hotel style

Return JSON only.
Return exactly 10 objects.
Each object must include:
destination
budget_fit
estimated_cost_level
reason

User profile:
{profile_json}

Destination options:
{destination_options_json}
"""

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )

    return response.text


# Validate and clean user profile
def create_user_profile(profile_json: str) -> str:
    profile = json.loads(profile_json)

    required_fields = [
        "origin",
        "age",
        "traveling_with_children",
        "number_of_travelers",
        "trip_length_days",
        "total_trip_budget",
        "destination_types",
        "hotel_budget_tier",
        "hotel_style",
        "pace",
        "crowd_tolerance"
    ]

    missing = [field for field in required_fields if field not in profile]

    if missing:
        raise ValueError(f"Missing profile fields: {missing}")

    profile["budget_note"] = "Budget is used for feasibility reasoning. Geoapify validates destination coordinates. Foursquare gives hotel and activity listings, not live booking prices."

    return json.dumps(profile, indent=2)


# Build itinerary object from selected items and assignments
def build_itinerary(profile_json: str, selected_destination: str, selected_hotel_json: str, selected_activities_json: str, assignments_json: str) -> str:
    profile = json.loads(profile_json)
    selected_hotel = json.loads(selected_hotel_json)
    selected_activities = json.loads(selected_activities_json)
    assignments = json.loads(assignments_json)

    trip_days = int(profile["trip_length_days"])

    itinerary = {
        "destination": selected_destination,
        "hotel": selected_hotel,
        "trip_length_days": trip_days,
        "daily_itinerary": {}
    }

    for day_num in range(1, trip_days + 1):
        itinerary["daily_itinerary"][f"Day {day_num}"] = {
            "Morning": [],
            "Afternoon": [],
            "Evening": []
        }

    for assignment in assignments:
        activity_index = int(assignment["activity_index"])
        day = assignment["day"]
        time_slot = assignment["time_slot"]

        if activity_index < 0 or activity_index >= len(selected_activities):
            raise ValueError("Activity index out of range.")

        if day not in itinerary["daily_itinerary"]:
            raise ValueError(f"{day} is outside the trip length.")

        if time_slot not in ["Morning", "Afternoon", "Evening"]:
            raise ValueError("Time slot must be Morning, Afternoon, or Evening.")

        itinerary["daily_itinerary"][day][time_slot].append(selected_activities[activity_index])

    return json.dumps(itinerary, indent=2)


# Tool registry
TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_datetime": get_datetime,
    "geocode_destination": geocode_destination,
    "recommend_destinations": recommend_destinations,
    "search_foursquare_places": search_foursquare_places,
    "search_places": search_places,
    "search_activity_options": search_activity_options,
    "estimate_budget_fit": estimate_budget_fit,
    "create_user_profile": create_user_profile,
    "build_itinerary": build_itinerary
}

# Quick tests
print("Calculator test:", calculator("247 * 19"))
print("Date/time test:", get_datetime())
print("Geoapify test:")
print(geocode_destination("Honolulu, Hawaii"))
print("Foursquare test:")
print(search_foursquare_places("Waikiki, Honolulu, Hawaii", "hotel", limit=2))

Calculator test: 4693
Date/time test: Tuesday, May 05, 2026, 05:05 PM local time
Geoapify test:
{
  "searched_destination": "Honolulu, Hawaii",
  "formatted_destination": "Honolulu, HI, United States of America",
  "city": "Honolulu",
  "state": "Hawaii",
  "country": "United States",
  "lat": 21.304547,
  "lon": -157.855676
}
Foursquare test:
{
  "query": "hotel",
  "destination": "Waikiki, Honolulu, Hawaii",
  "lat": 21.2793568,
  "lon": -157.8285713,
  "radius_meters": 70000,
  "results_count": 2,
  "places": [
    {
      "name": "The Surfjack Hotel & Swim Club",
      "address": "412 Lewers St, Honolulu, HI 96815",
      "city": "Honolulu",
      "state": "HI",
      "country": "US",
      "postcode": "96815",
      "categories": [
        "Hotel"
      ],
      "distance_meters": 301,
      "website": "http://surfjack.com",
      "fsq_place_id": "56b8eeba498e05a421cfffe2",
      "result_source": "Foursquare API",
      "source": "Foursquare API"
    },
    {
      "name": "Hawaii

###Cell 2 Output
Test

### Cell 2 Markdown: Safe Tool Execution

**Question:** Why is it important to restrict what `eval()` can execute? What could happen if you passed user input directly to an unrestricted `eval()`?

**Answer:** Restricting `eval()` matters because unrestricted user input could run dangerous Python code. A safe namespace limits the calculator to math operations only.

###Cell 3

### Cell 3 Tool Schemas

 tool descriptions to Gemini

| Tool schema | Description Gemini sees | Required input |
|---|---|---|
| `calculator` | Evaluates a mathematical expression and returns the numeric result. Used for travel budget arithmetic or any math operation. | `expression` |
| `get_datetime` | Returns the current date, time, and day of the week. Used for current time or date questions. | None |
| `geocode_destination` | Finds geographic information for a destination, including formatted place name, city, state, country, latitude, and longitude. | `destination` |
| `recommend_destinations` | Generates 10 destination recommendations from a user travel profile, then validates them with Geoapify geocoding. Used before destination selection. | `profile_json` |
| `search_foursquare_places` | Searches Foursquare for real hotels, businesses, attractions, beaches, trails, restaurants, spas, yoga studios, and activity places near a destination. Used for local travel place listings with names and addresses. | `destination`, `query` |
| `search_places` | Searches Foursquare for places near a destination using a category such as hotels, restaurants, spa, beach, museums, shopping, or nightlife. | `destination`, `category` |
| `search_geoapify_text` | Searches Geoapify using a text query near a selected destination. Used only as a backup utility, not for normal hotel or activity recommendations. | `query`, `destination` |
| `search_activity_options` | Searches for specific activity options near a destination based on selected activity categories. Uses Foursquare first and Gemini fallback only if fewer than 5 Foursquare results are found. | `destination`, `selected_activity_categories_json` |
| `estimate_budget_fit` | Estimates whether destination options fit the user's travel budget using general travel-cost reasoning. Does not use live prices. | `profile_json`, `destination_options_json` |
| `create_user_profile` | Validates and formats the user travel profile. | `profile_json` |
| `build_itinerary` | Builds a day-by-day itinerary from the selected destination, hotel, activities, and day/time assignments. | `profile_json`, `selected_destination`, `selected_hotel_json`, `selected_activities_json`, `assignments_json` |

In [6]:
# Cell 3: Define Tool Schemas for Gemini - tell Gemini what tools it can call

# Calculator schema
calculator_declaration = types.FunctionDeclaration(
    name="calculator",
    description="Evaluates a mathematical expression and returns the numeric result. Use this for travel budget arithmetic or any math operation.",
    parameters={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A math expression, such as '325 * 3'."
            }
        },
        "required": ["expression"]
    }
)

# Date/time schema
get_datetime_declaration = types.FunctionDeclaration(
    name="get_datetime",
    description="Returns the current date, time, and day of the week. Use this for current time or date questions.",
    parameters={
        "type": "object",
        "properties": {}
    }
)

# Geocode destination schema
geocode_destination_declaration = types.FunctionDeclaration(
    name="geocode_destination",
    description="Finds geographic information for a destination, including formatted place name, city, state, country, latitude, and longitude.",
    parameters={
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The destination to geocode, such as 'Waikiki, Honolulu, Hawaii'."
            }
        },
        "required": ["destination"]
    }
)

# Recommend destinations schema
recommend_destinations_declaration = types.FunctionDeclaration(
    name="recommend_destinations",
    description="Generates 10 destination recommendations from a user travel profile, then validates them with Geoapify geocoding. Use this before destination selection.",
    parameters={
        "type": "object",
        "properties": {
            "profile_json": {
                "type": "string",
                "description": "The user travel profile as a JSON string."
            },
            "required_destination": {
                "type": "string",
                "description": "A required destination that must appear in the recommendations."
            },
            "total_options": {
                "type": "integer",
                "description": "The number of destination options to return."
            }
        },
        "required": ["profile_json"]
    }
)

# Foursquare place search schema
search_foursquare_places_declaration = types.FunctionDeclaration(
    name="search_foursquare_places",
    description="Searches Foursquare for real hotels, businesses, attractions, beaches, trails, restaurants, spas, yoga studios, and activity places near a destination. Use this for local travel place listings with names and addresses.",
    parameters={
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "Destination near which to search, such as 'Waikiki, Honolulu, Hawaii'."
            },
            "query": {
                "type": "string",
                "description": "Search term such as hotel, yoga studio, beach, hiking trail, restaurant, spa, museum, or shopping."
            },
            "limit": {
                "type": "integer",
                "description": "Maximum number of results."
            },
            "radius_meters": {
                "type": "integer",
                "description": "Search radius in meters."
            }
        },
        "required": ["destination", "query"]
    }
)

# Search places schema
search_places_declaration = types.FunctionDeclaration(
    name="search_places",
    description="Searches Foursquare for places near a destination using a category such as hotels, restaurants, spa, beach, museums, shopping, or nightlife.",
    parameters={
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "Destination near which to search."
            },
            "category": {
                "type": "string",
                "description": "Place category, such as hotels, restaurants, spa, beach, or museums."
            },
            "limit": {
                "type": "integer",
                "description": "Maximum number of results."
            },
            "radius_meters": {
                "type": "integer",
                "description": "Search radius in meters."
            }
        },
        "required": ["destination", "category"]
    }
)

# Text search schema
search_geoapify_text_declaration = types.FunctionDeclaration(
    name="search_geoapify_text",
    description="Searches Geoapify using a text query near a selected destination. Use only as a backup utility, not for normal hotel or activity recommendations.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Text search query."
            },
            "destination": {
                "type": "string",
                "description": "Destination near which to search."
            },
            "limit": {
                "type": "integer",
                "description": "Maximum number of results."
            },
            "radius_meters": {
                "type": "integer",
                "description": "Search radius in meters."
            }
        },
        "required": ["query", "destination"]
    }
)

# Activity options schema
search_activity_options_declaration = types.FunctionDeclaration(
    name="search_activity_options",
    description="Searches for specific activity options near a destination based on selected activity categories. Uses Foursquare first and Gemini fallback only if fewer than 5 Foursquare results are found.",
    parameters={
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "Selected destination."
            },
            "selected_activity_categories_json": {
                "type": "string",
                "description": "JSON list of selected activity categories."
            },
            "limit_per_category": {
                "type": "integer",
                "description": "Number of results per activity category."
            }
        },
        "required": ["destination", "selected_activity_categories_json"]
    }
)

# Budget fit schema
estimate_budget_fit_declaration = types.FunctionDeclaration(
    name="estimate_budget_fit",
    description="Estimates whether destination options fit the user's travel budget using general travel-cost reasoning. This does not use live prices.",
    parameters={
        "type": "object",
        "properties": {
            "profile_json": {
                "type": "string",
                "description": "User travel profile as JSON."
            },
            "destination_options_json": {
                "type": "string",
                "description": "Destination options as JSON."
            }
        },
        "required": ["profile_json", "destination_options_json"]
    }
)

# Profile tool schema
create_user_profile_declaration = types.FunctionDeclaration(
    name="create_user_profile",
    description="Validates and formats the user travel profile.",
    parameters={
        "type": "object",
        "properties": {
            "profile_json": {
                "type": "string",
                "description": "User travel profile as JSON."
            }
        },
        "required": ["profile_json"]
    }
)

# Itinerary builder schema
build_itinerary_declaration = types.FunctionDeclaration(
    name="build_itinerary",
    description="Builds a day-by-day itinerary from the selected destination, hotel, activities, and day/time assignments.",
    parameters={
        "type": "object",
        "properties": {
            "profile_json": {
                "type": "string",
                "description": "User travel profile as JSON."
            },
            "selected_destination": {
                "type": "string",
                "description": "Selected destination."
            },
            "selected_hotel_json": {
                "type": "string",
                "description": "Selected hotel as JSON."
            },
            "selected_activities_json": {
                "type": "string",
                "description": "Selected activities as JSON."
            },
            "assignments_json": {
                "type": "string",
                "description": "Activity day/time assignments as JSON."
            }
        },
        "required": [
            "profile_json",
            "selected_destination",
            "selected_hotel_json",
            "selected_activities_json",
            "assignments_json"
        ]
    }
)

# Combine tool declarations
tools = [
    types.Tool(
        function_declarations=[
            calculator_declaration,
            get_datetime_declaration,
            geocode_destination_declaration,
            recommend_destinations_declaration,
            search_foursquare_places_declaration,
            search_places_declaration,
            search_geoapify_text_declaration,
            search_activity_options_declaration,
            estimate_budget_fit_declaration,
            create_user_profile_declaration,
            build_itinerary_declaration
        ]
    )
]

# Print tool names to verify
print("Tool declarations loaded:")
for declaration in tools[0].function_declarations:
    print("-", declaration.name)

Tool declarations loaded:
- calculator
- get_datetime
- geocode_destination
- recommend_destinations
- search_foursquare_places
- search_places
- search_geoapify_text
- search_activity_options
- estimate_budget_fit
- create_user_profile
- build_itinerary


###Cell 3 Output
Tool Declaration Recap

### Cell 3 Markdown: Tool Descriptions

**Question:** The model decides whether to call a tool based on the tool's description. How would you test whether your descriptions are good enough? What would happen if two tools had overlapping descriptions?

**Answer:** Tool descriptions can be tested by asking questions that should trigger each tool and checking the routing table. If two tools have overlapping descriptions, Gemini may choose the wrong tool or call a tool when it should answer directly.

### Cell 4: Agent Loop

| Step | What | Why |
|---|---|---|
| 1 | The user message is added to `chat_history`. | Keeps the conversation context. |
| 2 | Gemini receives the message, tool schemas, and system instruction. | Lets Gemini decide whether it needs a tool. |
| 3 | The code checks Gemini’s response for a tool call. | Separates direct answers from tool-using answers. |
| 4 | If Gemini calls a tool, the matching Python function from `TOOL_FUNCTIONS` runs. | Executes the actual work outside the model. |
| 5 | The tool result is sent back to Gemini. | Lets Gemini use the tool output in its final answer. |
| 6 | The final model response is added to `chat_history`. | Preserves the completed turn for later context. |
| 7 | Each tool call is saved in `LAST_TOOL_CALLS`. | Supports the audit table in Cell 9. |


In [7]:
# Cell 4: Agent Loop - Gemini chooses tools, Python runs them, and Gemini writes final response

# Store conversation turns
chat_history = []

# Store tool call logs
LAST_TOOL_CALLS = []

# Main behavior rules
SYSTEM_INSTRUCTION = """
You are a travel-planning assistant for an NLP lab about simple agents.

Use tools when the user needs current place data, hotel data, activity data, destination data, math, budget fit, or time information.
Do not invent hotel names, activity names, restaurants, attractions, or place data when API data is requested.

Use search_places for hotels and clear place categories.
Use search_activity_options for activity category recommendations near a destination.
Use search_geoapify_text only for a specific text search when no category tool fits.
Use recommend_destinations when the user needs destination recommendations from a profile.
Use estimate_budget_fit when the user needs destination budget fit reasoning.
Use calculator for math and budget arithmetic.
Use get_datetime for current date or time questions.

Budget fit uses general reasoning, not live prices.
Do not claim Geoapify provides live travel prices.
Geoapify provides place listings.

Destination type and activity interests are separate.
Activity categories should be shown only after destination/profile choices are known.

Do not build the CV crowd analyzer.
A selected beach activity can be marked as a later CV handoff if needed.

Keep responses short and structured.
If a tool returns no results, say that clearly instead of pretending it found results.
"""


# Extract text from Gemini response safely
def extract_text_from_response(response):
    text_parts = []

    try:
        if not response or not response.candidates:
            return ""

        content = response.candidates[0].content

        if not content or not content.parts:
            return ""

        for part in content.parts:
            if getattr(part, "text", None):
                text_parts.append(part.text)

    except Exception:
        return ""

    if text_parts:
        return "\n".join(text_parts)

    try:
        return response.text if hasattr(response, "text") and response.text else ""
    except Exception:
        return ""


# Get first function call from Gemini response safely
def get_first_function_call(response):
    try:
        if not response or not response.candidates:
            return None

        content = response.candidates[0].content

        if not content or not content.parts:
            return None

        for part in content.parts:
            if getattr(part, "function_call", None):
                return part.function_call

    except Exception:
        return None

    return None


# Run one agent turn
def agent_turn(user_message: str) -> str:
    global chat_history, LAST_TOOL_CALLS

    user_content = types.Content(
        role="user",
        parts=[types.Part(text=user_message)]
    )

    chat_history.append(user_content)

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=chat_history,
        config=types.GenerateContentConfig(
            tools=tools,
            system_instruction=SYSTEM_INSTRUCTION
        )
    )

    function_call = get_first_function_call(response)

    if function_call:
        function_name = function_call.name
        function_args = dict(function_call.args)

        if function_name not in TOOL_FUNCTIONS:
            tool_result = f"Tool error: unknown tool '{function_name}'."
        else:
            try:
                tool_result = TOOL_FUNCTIONS[function_name](**function_args)
            except Exception as e:
                tool_result = f"Tool error from {function_name}: {type(e).__name__}: {e}"

        LAST_TOOL_CALLS.append({
            "tool": function_name,
            "args": function_args,
            "result": tool_result
        })

        args_display = ", ".join([f"{k}={repr(v)}" for k, v in function_args.items()])
        print(f"[TOOL CALL] {function_name}({args_display}) -> {str(tool_result)[:500]}")

        try:
            if response.candidates and response.candidates[0].content:
                chat_history.append(response.candidates[0].content)
        except Exception:
            pass

        tool_response_content = types.Content(
            role="user",
            parts=[
                types.Part.from_function_response(
                    name=function_name,
                    response={"result": tool_result}
                )
            ]
        )

        chat_history.append(tool_response_content)

        try:
            final_response = client.models.generate_content(
                model=MODEL_ID,
                contents=chat_history,
                config=types.GenerateContentConfig(
                    tools=tools,
                    system_instruction=SYSTEM_INSTRUCTION
                )
            )

            final_text = extract_text_from_response(final_response)

        except Exception as e:
            final_text = f"The tool ran, but the final model response failed: {type(e).__name__}: {e}"

        if not final_text:
            final_text = f"Used tool {function_name}. Tool result preview: {str(tool_result)[:500]}"

        chat_history.append(
            types.Content(
                role="model",
                parts=[types.Part(text=final_text)]
            )
        )

        return final_text

    final_text = extract_text_from_response(response)

    if not final_text:
        final_text = "No text response was returned."

    chat_history.append(
        types.Content(
            role="model",
            parts=[types.Part(text=final_text)]
        )
    )

    return final_text


# Test agent loop
print(agent_turn("Find the coordinates for Honolulu, Hawaii."))

[TOOL CALL] geocode_destination(destination='Honolulu, Hawaii') -> {
  "searched_destination": "Honolulu, Hawaii",
  "formatted_destination": "Honolulu, HI, United States of America",
  "city": "Honolulu",
  "state": "Hawaii",
  "country": "United States",
  "lat": 21.304547,
  "lon": -157.855676
}
The coordinates for Honolulu, Hawaii are:
* **Latitude:** 21.304547
* **Longitude:** -157.855676


###Cell 4 Output
Test

### Cell 4 Markdown: Agent Loop Data Flow

**Question:** Trace the data flow for a single tool-using turn. How many API calls did it take? Why does the model need to be called twice — once to decide to use the tool, and again to incorporate the tool's result?

**Answer:** In the Cell 4 test, the user asked for Honolulu’s location. Gemini first received the user message and selected `geocode_destination(destination="Honolulu, Hawaii")`. The Python code ran that tool, and Geoapify returned the formatted destination plus latitude `21.304547` and longitude `-157.855676`. The tool result was then sent back to Gemini, and Gemini used it to write the final answer. This took three API calls total including the Geoapify API call buy two Gemini API calls: one to choose the tool and one to explain the tool result.

### Cell 5: User Profile and Destination Choice

| Step | What | Why |
|---|---|---|
| 1 | The user enters trip details such as origin, age, budget, trip length, destination interests, hotel tier, hotel style, pace, and crowd tolerance. | Builds the travel profile used for recommendations. |
| 2 | `create_user_profile()` validates the required profile fields. | Makes sure the workflow has the required inputs before moving forward. |
| 3 | `recommend_destinations()` generates 10 destination options from the profile. | Creates destination recommendations based on the user’s preferences. |
| 4 | Geoapify validates the destination names and coordinates. | Confirms the recommendations are searchable locations. |
| 5 | `estimate_budget_fit()` reviews the 10 destinations against the user’s budget. | Adds general budget reasoning without claiming live travel prices. |
| 6 | The notebook prints a formatted user profile and destination table. | Makes the output easier to review and present. |
| 7 | The user chooses one destination. | Sets the selected location for hotel and activity searches. |

In [8]:
# Cell 5: Build Profile + Generate Destination Recommendations + Choose Destination

# Start workflow tool log for later audit
workflow_tool_log = []

# Destination type options for profile input
destination_type_options = [
    "beach",
    "mountain",
    "wellness/spa",
    "metropolitan/city",
    "physical/adventure",
    "skiing/snowboarding",
    "national parks",
    "food/culinary",
    "culture/history",
    "shopping",
    "nightlife",
    "family-friendly",
    "romantic/couples",
    "luxury/resort",
    "eco/nature"
]

# Hotel budget options for profile input
hotel_budget_options = {
    "$": "Select Value",
    "$$": "Comfort",
    "$$$": "Premium",
    "$$$$": "Luxury",
    "$$$$$": "Ultra-Luxury"
}

# Hotel style options for profile input
hotel_style_options = [
    "standard hotel",
    "resort",
    "boutique hotel",
    "family-friendly hotel",
    "luxury hotel",
    "eco-friendly hotel",
    "walkable city hotel"
]

# Print profile form
print("=" * 80)
print("BUILD YOUR TRAVEL PROFILE")
print("=" * 80)

# Ask profile questions
origin = input("Where are you traveling from? Example: OKC, Oklahoma City, or Dallas: ")
age = input("What is your age? ")
traveling_with_children = input("Are you traveling with children? yes/no: ")
number_of_travelers = int(input("How many total travelers? "))
trip_length_days = int(input("How many days is the trip? "))
total_trip_budget = float(input("What is your total trip budget? Example: 5000: "))

# Show destination type options
print("\nDESTINATION TYPE OPTIONS")
for i, option in enumerate(destination_type_options, start=1):
    print(f"{i}. {option}")

# User selects destination types
destination_type_numbers = input("\nChoose destination types by number, separated by commas: ")
selected_destination_type_numbers = parse_number_list(destination_type_numbers)

selected_destination_types = [
    destination_type_options[i - 1]
    for i in selected_destination_type_numbers
    if 1 <= i <= len(destination_type_options)
]

# Show hotel budget options
print("\nHOTEL BUDGET OPTIONS")
for key, value in hotel_budget_options.items():
    print(f"{key}: {value}")

# User selects hotel budget tier
hotel_budget_tier = input("\nChoose hotel budget tier. Example: $$$: ")

# Show hotel style options
print("\nHOTEL STYLE OPTIONS")
for i, option in enumerate(hotel_style_options, start=1):
    print(f"{i}. {option}")

# User selects hotel style
hotel_style_number = int(input("\nChoose hotel style number: "))
hotel_style = hotel_style_options[hotel_style_number - 1]

# Ask remaining preferences
pace = input("Preferred pace? relaxed, balanced, or packed: ")
crowd_tolerance = input("Crowd tolerance? low, medium, or high: ")

# Build profile before destination selection
profile_without_destination = {
    "origin": origin,
    "age": age,
    "traveling_with_children": traveling_with_children,
    "number_of_travelers": number_of_travelers,
    "trip_length_days": trip_length_days,
    "total_trip_budget": total_trip_budget,
    "destination_types": selected_destination_types,
    "hotel_budget_tier": hotel_budget_tier,
    "hotel_style": hotel_style,
    "pace": pace,
    "crowd_tolerance": crowd_tolerance
}

# Print formatted profile
print("\n" + "=" * 80)
print("TRAVEL PROFILE")
print("=" * 80)
print(f"Origin: {origin}")
print(f"Age: {age}")
print(f"Traveling with children: {traveling_with_children}")
print(f"Travelers: {number_of_travelers}")
print(f"Trip length: {trip_length_days} days")
print(f"Total trip budget: ${total_trip_budget:,.0f}")
print(f"Destination types: {', '.join(selected_destination_types)}")
print(f"Hotel budget tier: {hotel_budget_tier} {hotel_budget_options.get(hotel_budget_tier, '')}")
print(f"Hotel style: {hotel_style}")
print(f"Pace: {pace}")
print(f"Crowd tolerance: {crowd_tolerance}")
print("=" * 80)

# Generate profile-based destination recommendations
destination_recommendation_result = recommend_destinations(
    profile_json=json.dumps(profile_without_destination),
    required_destination="Waikiki, Honolulu, Hawaii",
    total_options=10
)

# Log destination recommendation tool usage
workflow_tool_log.append({
    "workflow_step": "Profile-based destination recommendations",
    "expected_tool": "recommend_destinations",
    "actual_tool": "recommend_destinations",
    "tool_type": "Direct Python tool call",
    "notes": "Gemini generated destination options from the user profile, then Geoapify validated/geocoded them."
})

# Load destination recommendation results
destination_recommendation_data = json.loads(destination_recommendation_result)
destination_options_data = destination_recommendation_data["destinations"]

# Convert validated destinations into simple list for budget scoring
destination_options = [
    item["destination"]
    for item in destination_options_data
]

# Score destinations for budget fit
budget_fit_result = estimate_budget_fit(
    profile_json=json.dumps(profile_without_destination),
    destination_options_json=json.dumps(destination_options)
)

# Log budget tool usage
workflow_tool_log.append({
    "workflow_step": "Budget-fit destination recommendations",
    "expected_tool": "estimate_budget_fit",
    "actual_tool": "estimate_budget_fit",
    "tool_type": "Direct Python tool call",
    "notes": "Gemini estimated budget fit because Geoapify does not provide live travel prices."
})

# Clean Gemini JSON output
clean_budget_text = budget_fit_result.strip()
clean_budget_text = clean_budget_text.replace("```json", "").replace("```", "").strip()

# Parse budget fit output
try:
    budget_fit_data = json.loads(clean_budget_text)
except Exception:
    print("Gemini budget output could not be parsed as JSON. Raw output:")
    print(budget_fit_result)

    budget_fit_data = [
        {
            "destination": d,
            "budget_fit": "unknown",
            "estimated_cost_level": "unknown",
            "reason": "Budget fit could not be parsed."
        }
        for d in destination_options
    ]

# Print destination choices
print("\n" + "=" * 80)
print("DESTINATION OPTIONS GENERATED FROM YOUR PROFILE")
print("=" * 80)

for i, item in enumerate(budget_fit_data, start=1):
    print(f"{i}. {item.get('destination')}")
    print(f"   Budget fit: {item.get('budget_fit')}")
    print(f"   Estimated cost level: {item.get('estimated_cost_level')}")
    print(f"   Reason: {item.get('reason')}")

    matching_validation = [
        d for d in destination_options_data
        if d["destination"] == item.get("destination")
    ]

    if matching_validation:
        print(f"   Validation source: {matching_validation[0].get('validation_source')}")

    print()

# User chooses destination
destination_choice_number = int(input("Choose one destination number: "))
selected_destination = budget_fit_data[destination_choice_number - 1]["destination"]

# Build final profile with selected destination
user_profile = profile_without_destination.copy()
user_profile["selected_destination"] = selected_destination

# Validate profile with profile tool
profile_json = create_user_profile(json.dumps(user_profile))

# Log profile tool usage
workflow_tool_log.append({
    "workflow_step": "User profile validation",
    "expected_tool": "create_user_profile",
    "actual_tool": "create_user_profile",
    "tool_type": "Direct Python tool call",
    "notes": "Validated required profile fields and saved the selected destination."
})

# Print selected destination
print("\n" + "=" * 80)
print("DESTINATION SELECTED")
print("=" * 80)
print(selected_destination)

BUILD YOUR TRAVEL PROFILE
Where are you traveling from? Example: OKC, Oklahoma City, or Dallas: Oklahoma City
What is your age? 38
Are you traveling with children? yes/no: No
How many total travelers? 2
How many days is the trip? 5
What is your total trip budget? Example: 5000: 6000

DESTINATION TYPE OPTIONS
1. beach
2. mountain
3. wellness/spa
4. metropolitan/city
5. physical/adventure
6. skiing/snowboarding
7. national parks
8. food/culinary
9. culture/history
10. shopping
11. nightlife
12. family-friendly
13. romantic/couples
14. luxury/resort
15. eco/nature

Choose destination types by number, separated by commas: 1,3,5,8,13,14

HOTEL BUDGET OPTIONS
$: Select Value
$$: Comfort
$$$: Premium
$$$$: Luxury
$$$$$: Ultra-Luxury

Choose hotel budget tier. Example: $$$: $$$

HOTEL STYLE OPTIONS
1. standard hotel
2. resort
3. boutique hotel
4. family-friendly hotel
5. luxury hotel
6. eco-friendly hotel
7. walkable city hotel

Choose hotel style number: 2
Preferred pace? relaxed, balanced, o

###Cell 5 Ouptut
User Profile and Destination

### Cell 6: Hotel and Activity Category Choice

| Step | What | Why |
|---|---|---|
| 1 | The selected destination from Cell 5 is printed. | Confirms the trip location before searching hotels. |
| 2 | `search_places()` searches Foursquare for hotel options near the selected destination. | Pulls real hotel/place listings. |
| 3 | The notebook prints hotel names, addresses, websites, and sources. | Lets the user compare hotel options. |
| 4 | The user chooses one hotel. | Sets the lodging choice for the itinerary. |
| 5 | Activity categories are generated from the destination type in the user profile. | Keeps activity interests tied to the selected trip style. |
| 6 | The user chooses broad activity categories. | Narrows the later activity search to the user’s interests. |

In [9]:

# Cell 6: Choose Hotel + Broad Activity Categories - select hotel and select activity categories

# Activity options depend on destination type
activity_options_by_destination_type = {
    "beach": [
        "beach time",
        "snorkeling",
        "surfing",
        "paddleboarding",
        "sunset cruise",
        "boat tour",
        "beachfront dining",
        "coastal walk",
        "spa day"
    ],
    "mountain": [
        "hiking",
        "scenic drive",
        "wildlife viewing",
        "lake activities",
        "mountain biking",
        "ziplining",
        "campfire dinner",
        "stargazing",
        "spa day"
    ],
    "wellness/spa": [
        "spa treatment",
        "massage",
        "yoga class",
        "healthy dining",
        "meditation",
        "thermal baths",
        "nature walk"
    ],
    "metropolitan/city": [
        "museums",
        "restaurants",
        "shopping",
        "theater",
        "nightlife",
        "walking tour",
        "architecture tour",
        "rooftop bar",
        "live music"
    ],
    "physical/adventure": [
        "hiking",
        "rock climbing",
        "mountain biking",
        "kayaking",
        "ziplining",
        "ATV tour",
        "trail running",
        "guided adventure tour"
    ],
    "skiing/snowboarding": [
        "skiing",
        "snowboarding",
        "snowshoeing",
        "hot springs",
        "apres-ski dining",
        "scenic gondola",
        "winter hiking"
    ],
    "national parks": [
        "scenic overlooks",
        "hiking",
        "wildlife viewing",
        "ranger-led tour",
        "nature photography",
        "camping",
        "stargazing",
        "visitor center"
    ],
    "food/culinary": [
        "food tour",
        "local restaurants",
        "farmers market",
        "cooking class",
        "street food",
        "coffee shops",
        "dessert spots"
    ],
    "culture/history": [
        "museums",
        "historic sites",
        "local landmarks",
        "cultural tour",
        "art galleries",
        "architecture tour",
        "heritage walk"
    ],
    "shopping": [
        "boutiques",
        "local markets",
        "shopping districts",
        "souvenir shops",
        "artisan markets",
        "luxury shopping"
    ],
    "nightlife": [
        "bars",
        "live music",
        "clubs",
        "comedy show",
        "rooftop lounge",
        "late-night dining"
    ],
    "family-friendly": [
        "aquarium",
        "zoo",
        "parks",
        "easy walking tour",
        "beach time",
        "interactive museum",
        "family restaurants"
    ],
    "romantic/couples": [
        "sunset dinner",
        "couples spa",
        "scenic walk",
        "boat tour",
        "wine tasting",
        "private tour",
        "beach picnic"
    ],
    "luxury/resort": [
        "resort pool",
        "spa day",
        "fine dining",
        "private cabana",
        "golf",
        "boutique shopping",
        "sunset cruise"
    ],
    "eco/nature": [
        "nature walk",
        "wildlife viewing",
        "eco tour",
        "botanical garden",
        "kayaking",
        "birdwatching",
        "conservation tour"
    ]
}

# Print selected destination
print("=" * 80)
print("SELECTED DESTINATION")
print("=" * 80)
print(selected_destination)

# Search hotels with Foursquare tool
print("\n" + "=" * 80)
print("HOTEL OPTIONS")
print("=" * 80)

hotel_result = search_places(
    destination=selected_destination,
    category="hotels",
    limit=7
)

# Log hotel tool usage
workflow_tool_log.append({
    "workflow_step": "Hotel options for selected destination",
    "expected_tool": "search_places",
    "actual_tool": "search_places",
    "tool_type": "Direct Python tool call",
    "notes": "Used Foursquare place listings to retrieve hotel options near the selected destination."
})

# Load hotel results
hotel_data = json.loads(hotel_result)
hotel_options = hotel_data["places"]

# Print hotel options
for i, hotel in enumerate(hotel_options, start=1):
    print(f"{i}. {hotel.get('name')}")
    print(f"   Address: {hotel.get('address')}")
    print(f"   Website: {hotel.get('website')}")
    print(f"   Source: {hotel.get('result_source') or hotel.get('source')}")
    print()

# User chooses hotel
hotel_choice_number = int(input("Choose one hotel number: "))
selected_hotel = hotel_options[hotel_choice_number - 1]

# Build activity category options from selected destination types
available_activity_categories = []

for destination_type in user_profile["destination_types"]:
    available_activity_categories.extend(activity_options_by_destination_type.get(destination_type, []))

# Remove duplicates while preserving order
available_activity_categories = list(dict.fromkeys(available_activity_categories))

# Print selected hotel
print("\n" + "=" * 80)
print("HOTEL SELECTED")
print("=" * 80)
print(selected_hotel.get("name"))
print(selected_hotel.get("address"))

# Print activity category options
print("\n" + "=" * 80)
print("ACTIVITY CATEGORY OPTIONS BASED ON YOUR PROFILE")
print("=" * 80)

for i, option in enumerate(available_activity_categories, start=1):
    print(f"{i}. {option}")

# User chooses broad activity categories
activity_category_numbers = input("\nChoose activity categories by number, separated by commas: ")
selected_activity_category_numbers = parse_number_list(activity_category_numbers)

selected_activity_categories = [
    available_activity_categories[i - 1]
    for i in selected_activity_category_numbers
    if 1 <= i <= len(available_activity_categories)
]

# Print selected activity categories
print("\n" + "=" * 80)
print("SELECTED ACTIVITY CATEGORIES")
print("=" * 80)

for category in selected_activity_categories:
    print("-", category)

SELECTED DESTINATION
Waikīkī, Honolulu, HI, United States of America

HOTEL OPTIONS
1. The Surfjack Hotel & Swim Club
   Address: 412 Lewers St, Honolulu, HI 96815
   Website: http://surfjack.com
   Source: Foursquare API

2. Hawaii Prince Hotel Waikiki and Golf Club
   Address: 100 Holomoana St (at Ala Moana Blvd), Honolulu, HI 96815
   Website: http://www.princeresortshawaii.com/hawaii-prince-hotel-waikiki/index.php
   Source: Foursquare API

3. Kaimana Beach Hotel
   Address: 2863 Kalakaua Ave (btwn Monsarrat Ave. Poni Moi Rd.), Honolulu, HI 96815
   Website: https://www.kaimana.com
   Source: Foursquare API

4. Park Shore Waikiki Hotel
   Address: 2586 Kalakaua Ave (Kapahulu Ave.), Honolulu, HI 96815
   Website: https://www.parkshorewaikiki.com
   Source: Foursquare API

5. Hale Koa Hotel
   Address: 2055 Kalia Rd, Honolulu, HI 96815
   Website: http://www.halekoa.com
   Source: Foursquare API

6. OUTRIGGER Waikīkī Paradise Hotel
   Address: 150 Kaiulani Ave (at Kuhio Ave.), Honolu

###Cell 6 Output
Hotel and Activity Interests

### Cell 7: Specific Activity Search and Selection

| Step | What | Why |
|---|---|---|
| 1 | The notebook prints the selected destination, hotel, and broad activity categories. | Confirms the trip context before searching. |
| 2 | `search_activity_options()` searches Foursquare for 5 specific options per selected activity category. | Pulls real local activity/place listings tied to the user’s interests. |
| 3 | Gemini fallback fills missing spots only if Foursquare returns fewer than 5 results. | Keeps the list complete while labeling non-API results clearly. |
| 4 | The notebook prints specific activity names, addresses, categories, and sources. | Makes the recommendations reviewable. |
| 5 | The user chooses which specific activities to add to the itinerary. | Sends only selected activities to the itinerary builder. |

In [11]:
# Cell 7: Pull Specific Activities + Choose Activities - fetch 5 live/fallback results per selected category and choose specific activities

# Print context from previous cells
print("=" * 80)
print("TRIP CONTEXT")
print("=" * 80)
print(f"Destination: {selected_destination}")
print(f"Hotel: {selected_hotel.get('name')}")
print("Selected activity categories:", ", ".join(selected_activity_categories))

# Pull 5 results per selected activity category
activity_result = search_activity_options(
    destination=selected_destination,
    selected_activity_categories_json=json.dumps(selected_activity_categories),
    limit_per_category=5
)

# Log activity tool usage
workflow_tool_log.append({
    "workflow_step": "Specific activity options from selected categories",
    "expected_tool": "search_activity_options",
    "actual_tool": "search_activity_options",
    "tool_type": "Direct Python tool call",
    "notes": "Used Foursquare first and Gemini fallback only when fewer than 5 Foursquare results were available."
})

# Load activity results
activity_data = json.loads(activity_result)
results_by_category = activity_data["results_by_category"]

# Flatten activity options into one numbered list
specific_activity_options = []

print("\n" + "=" * 80)
print("SPECIFIC ACTIVITY OPTIONS")
print("=" * 80)

option_number = 1

for category, places in results_by_category.items():
    print(f"\n{category.upper()}")

    for place in places:
        place["matched_category"] = category
        place["matched_activity_category"] = place.get("matched_activity_category", category)

        specific_activity_options.append(place)

        print(f"{option_number}. {place.get('name')}")
        print(f"   Address: {place.get('address')}")
        print(f"   Category: {category}")
        print(f"   Source: {place.get('result_source') or place.get('source')}")
        print()

        option_number += 1

# User chooses specific activities
specific_activity_numbers = input("Choose specific activities to add to the itinerary, separated by commas: ")
selected_specific_activity_numbers = parse_number_list(specific_activity_numbers)

selected_specific_activities = [
    specific_activity_options[i - 1]
    for i in selected_specific_activity_numbers
    if 1 <= i <= len(specific_activity_options)
]

# Print selected activities
print("\n" + "=" * 80)
print("SELECTED SPECIFIC ACTIVITIES")
print("=" * 80)

for i, activity in enumerate(selected_specific_activities, start=1):
    print(f"{i}. {activity.get('name')}")
    print(f"   Category: {activity.get('matched_activity_category') or activity.get('matched_category')}")
    print(f"   Address: {activity.get('address')}")
    print(f"   Source: {activity.get('result_source') or activity.get('source')}")
    print()

TRIP CONTEXT
Destination: Waikīkī, Honolulu, HI, United States of America
Hotel: Park Shore Waikiki Hotel
Selected activity categories: beach time, spa treatment, yoga class, healthy dining, hiking, fine dining, private cabana

SPECIFIC ACTIVITY OPTIONS

BEACH TIME
1. Waikiki Beach
   Address: Waikiki, Honolulu, Hawaii
   Category: beach time
   Source: Required project handoff option

2. Waikīkī Beach
   Address: Kalākaua Ave, Honolulu, HI 96815
   Category: beach time
   Source: Foursquare API

3. The Beach Bar
   Address: 2365 Kalakaua Ave (at Kaiulani Ave), Honolulu, HI 96815
   Category: beach time
   Source: Foursquare API

4. Kaimana Beach Park
   Address: 2863 Kalakaua Ave, Honolulu, HI 96815
   Category: beach time
   Source: Foursquare API

5. Kuhio Beach Park
   Address: Kalakaua Ave, Honolulu, HI 86813
   Category: beach time
   Source: Foursquare API


SPA TREATMENT
6. Waikiki Beach Marriott Resort & Spa
   Address: 2552 Kalakaua Ave (btwn Ohua Ave. & Paokalani Ave.), Hono

###Cell 7 Output
Activity Choices for Itinerary

### Cell 8: Itinerary Placement and Build

| Step | What | Why |
|---|---|---|
| 1 | The notebook shows each selected activity one at a time. | Lets the user decide where each activity belongs. |
| 2 | The user enters a day number and time slot abbreviation: `M`, `A`, or `E`. | Assigns activities to Morning, Afternoon, or Evening. |
| 3 | Blank input skips an activity. | The itinerary does not require every selected activity to be scheduled. |
| 4 | `build_itinerary()` creates the structured day-by-day itinerary. | Converts user choices into a itinerary. |
| 5 | Invalid day or time choices are rejected by the itinerary tool. |

In [13]:
# Cell 8: Assign Activities to Itinerary Slots - place each activity one at a time using short day/time codes

# Helper to parse placement input like 1M, 1A, 1E, day 1 M, Day 2 afternoon
def parse_single_activity_placement(placement_text, trip_length_days):
    placement_text = str(placement_text).strip().lower()

    if placement_text == "":
        return None

    day_match = re.search(r"\d+", placement_text)

    if not day_match:
        raise ValueError("Enter a day number and time code, such as 1M, 2A, or 3E.")

    day_number = int(day_match.group())

    if day_number < 1 or day_number > trip_length_days:
        raise ValueError(f"Day {day_number} is outside the {trip_length_days}-day trip.")

    if re.search(r"\bm\b", placement_text) or "morning" in placement_text or placement_text.endswith("m"):
        time_slot = "Morning"
    elif re.search(r"\ba\b", placement_text) or "afternoon" in placement_text or placement_text.endswith("a"):
        time_slot = "Afternoon"
    elif re.search(r"\be\b", placement_text) or "evening" in placement_text or "night" in placement_text or placement_text.endswith("e"):
        time_slot = "Evening"
    else:
        raise ValueError("Use M for morning, A for afternoon, or E for evening.")

    return {
        "day": f"Day {day_number}",
        "time_slot": time_slot
    }


# Pull trip length from profile
trip_length_days = int(user_profile["trip_length_days"])

# Print selected trip details
print("=" * 80)
print("ASSIGN ACTIVITIES TO YOUR ITINERARY")
print("=" * 80)
print(f"Destination: {selected_destination}")
print(f"Hotel: {selected_hotel.get('name')}")
print(f"Trip length: {trip_length_days} days")

print("\nFor each activity, enter the day number plus time code.")
print("Examples:")
print("1M = Day 1 Morning")
print("1A = Day 1 Afternoon")
print("1E = Day 1 Evening")
print("day 2 A = Day 2 Afternoon")
print("\nPress Enter with no text to skip an activity.")
print("Time codes: M = Morning, A = Afternoon, E = Evening")

# Store itinerary assignments
itinerary_assignments = []

# Ask placement one activity at a time
for i, activity in enumerate(selected_specific_activities, start=1):
    print("\n" + "-" * 80)
    print(f"{i}. {activity.get('name')}")
    print(f"   Category: {activity.get('matched_activity_category') or activity.get('matched_category')}")
    print(f"   Address: {activity.get('address')}")
    print(f"   Source: {activity.get('result_source') or activity.get('source')}")

    placement_text = input("Placement for this activity, or press Enter to skip: ")

    try:
        placement = parse_single_activity_placement(
            placement_text=placement_text,
            trip_length_days=trip_length_days
        )

        if placement is None:
            print("Skipped.")
            continue

        itinerary_assignments.append({
            "activity_index": i - 1,
            "activity_name": activity.get("name"),
            "day": placement["day"],
            "time_slot": placement["time_slot"]
        })

        print(f"Saved: {activity.get('name')} -> {placement['day']}, {placement['time_slot']}")

    except Exception as e:
        print(f"Skipped because the placement was not valid: {e}")


# Use itinerary-builder tool to create final itinerary
final_itinerary_json = build_itinerary(
    profile_json=json.dumps(user_profile),
    selected_destination=selected_destination,
    selected_hotel_json=json.dumps(selected_hotel),
    selected_activities_json=json.dumps(selected_specific_activities),
    assignments_json=json.dumps(itinerary_assignments)
)

# Log itinerary builder tool usage
workflow_tool_log.append({
    "workflow_step": "Final itinerary construction",
    "expected_tool": "build_itinerary",
    "actual_tool": "build_itinerary",
    "tool_type": "Direct Python tool call",
    "notes": "Built the final itinerary from selected activities and day/time assignments."
})

# Print placement check
print("\n" + "=" * 80)
print("PLACEMENT CHECK")
print("=" * 80)

if not itinerary_assignments:
    print("No activities were added to the itinerary.")
else:
    for assignment in itinerary_assignments:
        print(f"{assignment['day']} - {assignment['time_slot']}: {assignment['activity_name']}")

print("\nCell 8 finished. Run Cell 9 for the audit tables, then Cell 10 for the final itinerary.")

ASSIGN ACTIVITIES TO YOUR ITINERARY
Destination: Waikīkī, Honolulu, HI, United States of America
Hotel: Park Shore Waikiki Hotel
Trip length: 5 days

For each activity, enter the day number plus time code.
Examples:
1M = Day 1 Morning
1A = Day 1 Afternoon
1E = Day 1 Evening
day 2 A = Day 2 Afternoon

Press Enter with no text to skip an activity.
Time codes: M = Morning, A = Afternoon, E = Evening

--------------------------------------------------------------------------------
1. Waikiki Beach
   Category: beach time
   Address: Waikiki, Honolulu, Hawaii
   Source: Required project handoff option
Placement for this activity, or press Enter to skip: 2A
Saved: Waikiki Beach -> Day 2, Afternoon

--------------------------------------------------------------------------------
2. Mandara Spa & Salon
   Category: spa treatment
   Address: in Kālia Tower, Hilton Hawaiian Village (2005 Kalia Rd), Honolulu, HI 96815
   Source: Foursquare API
Placement for this activity, or press Enter to skip: 

###Cell 8 Output
Activity Time Placement for Itinerary

### Cell 9: Tool Routing, Recommendation Quality, and Edge-Case Audit

| Step | What | Why |
|---|---|---|
| 1 | The workflow tool audit checks which tools were used during the travel workflow. | Shows whether the notebook used the expected tools for each step. |
| 2 | The agent routing test asks Gemini different types of questions. | Tests whether Gemini can choose the correct tool on its own. |
| 3 | The routing accuracy table compares expected tools to actual tools. | Measures tool-routing accuracy. |
| 4 | The recommendation quality audit checks source type, coverage, and local relevance. | Shows whether recommendations came from the API, Gemini fallback, or required project handoff. |
| 5 | The activity source breakdown shows results by category. | Identifies which activity categories were strong or weak. |
| 6 | Travel edge-case tests check bad inputs and weak search categories. | Tests whether the workflow handles failures without crashing. |
| 7 | The final audit summary reports the main scores. | Gives a quick performance snapshot. |

In [17]:
# Cell 9: Tool Routing + Recommendation Quality + Travel Edge-Case Audit

# Helper to safely count results
def safe_len(value):
    try:
        return len(value)
    except Exception:
        return 0


# Helper to check if a place appears local to selected destination
def is_likely_local_to_selected_destination(place, selected_destination):
    destination_text = selected_destination.lower()

    place_text = " ".join([
        str(place.get("name", "")),
        str(place.get("address", "")),
        str(place.get("city", "")),
        str(place.get("state", "")),
        str(place.get("country", ""))
    ]).lower()

    if (
        "hawaii" in destination_text
        or "honolulu" in destination_text
        or "waikiki" in destination_text
        or "oahu" in destination_text
    ):
        return (
            "hawaii" in place_text
            or "honolulu" in place_text
            or "waikiki" in place_text
            or "oahu" in place_text
            or place.get("result_source") in ["Gemini fallback", "Required project handoff option"]
            or place.get("source") in ["Gemini fallback", "Required project handoff option"]
        )

    return bool(place.get("address") or place.get("city") or place.get("state") or place.get("country"))


# Helper to check whether a tool result has usable records
def tool_result_is_usable(actual_tool, tool_result, answer_text=""):
    if actual_tool == "none":
        return bool(answer_text)

    if actual_tool in ["calculator", "get_datetime", "estimate_budget_fit", "create_user_profile", "build_itinerary"]:
        return bool(tool_result)

    try:
        data = json.loads(tool_result)

        if actual_tool == "recommend_destinations":
            return len(data.get("destinations", [])) == 10

        if "places" in data:
            return len(data.get("places", [])) > 0

        if "results_by_category" in data:
            total = 0

            for places in data.get("results_by_category", {}).values():
                total += len(places)

            return total > 0

        return bool(data)

    except Exception:
        return False


# Build workflow tool audit table from actual workflow log
workflow_tool_audit_df = pd.DataFrame(workflow_tool_log)

# Recompute correctness instead of trusting a stored value
workflow_tool_audit_df["correct"] = workflow_tool_audit_df["expected_tool"] == workflow_tool_audit_df["actual_tool"]

# Calculate workflow tool accuracy
workflow_tool_accuracy = workflow_tool_audit_df["correct"].mean()

# Print workflow tool audit
print("=" * 80)
print("WORKFLOW TOOL AUDIT")
print("=" * 80)
print("This table checks whether each major workflow step used the expected workflow tool.")
display(workflow_tool_audit_df)

print("Workflow tool accuracy:", round(workflow_tool_accuracy, 3))


# Agent routing tests required by lab, adapted to travel theme
# These prompts do not tell Gemini which tool name to use.
routing_test_questions = [
    {
        "question": f"I need 10 destination ideas for this travel profile. Include Waikiki as one option for a later beach crowd analysis: {json.dumps(profile_without_destination)}",
        "expected_tool": "recommend_destinations"
    },
    {
        "question": "A hotel costs 325 dollars per night for 3 nights. What is the hotel subtotal?",
        "expected_tool": "calculator"
    },
    {
        "question": "What time is it right now?",
        "expected_tool": "get_datetime"
    },
    {
        "question": f"Find hotel options near {selected_destination}.",
        "expected_tool": "search_places"
    },
    {
        "question": f"What specific activity options can I do near {selected_destination} for these interests: {selected_activity_categories}?",
        "expected_tool": "search_activity_options"
    },
    {
        "question": "Write one short tagline for a travel-planning app.",
        "expected_tool": "none"
    }
]

# Store agent routing results
agent_routing_results = []

# Run each routing test in a fresh conversation
for test in routing_test_questions:
    chat_history = []
    LAST_TOOL_CALLS = []

    question = test["question"]
    expected_tool = test["expected_tool"]

    print("\n" + "=" * 80)
    print("AGENT ROUTING TEST")
    print("Question:", question[:500])

    try:
        answer = agent_turn(question)

        actual_tool = LAST_TOOL_CALLS[0]["tool"] if LAST_TOOL_CALLS else "none"
        tool_result = LAST_TOOL_CALLS[0]["result"] if LAST_TOOL_CALLS else ""

        correct_tool = expected_tool == actual_tool
        usable_result = tool_result_is_usable(
            actual_tool=actual_tool,
            tool_result=tool_result,
            answer_text=answer
        )

        agent_routing_results.append({
            "question": question[:120],
            "expected_tool": expected_tool,
            "actual_tool": actual_tool,
            "correct_tool": correct_tool,
            "usable_result": usable_result,
            "fully_successful": correct_tool and usable_result
        })

        print("Expected tool:", expected_tool)
        print("Actual tool:", actual_tool)
        print("Correct tool:", correct_tool)
        print("Usable result:", usable_result)

    except Exception as e:
        agent_routing_results.append({
            "question": question[:120],
            "expected_tool": expected_tool,
            "actual_tool": "crashed",
            "correct_tool": False,
            "usable_result": False,
            "fully_successful": False
        })

        print("Routing test crashed:", e)

    time.sleep(1)

# Build routing dataframe
agent_routing_df = pd.DataFrame(agent_routing_results)

# Calculate routing accuracy and full success rate
agent_routing_accuracy = agent_routing_df["correct_tool"].mean()
agent_routing_success_rate = agent_routing_df["fully_successful"].mean()

# Print routing accuracy table
print("\n" + "=" * 80)
print("AGENT ROUTING ACCURACY TABLE")
print("=" * 80)
print("This table checks whether Gemini chose the expected tool without being told the tool name.")
display(agent_routing_df)

print("Agent routing accuracy:", round(agent_routing_accuracy, 3))
print("Agent routing full success rate:", round(agent_routing_success_rate, 3))


# Recommendation quality audit rows
recommendation_quality_rows = []

# Destination recommendation quality
try:
    destination_count = safe_len(budget_fit_data)
    validated_destination_count = safe_len(destination_options_data)

    recommendation_quality_rows.append({
        "recommendation_area": "Destination recommendations",
        "items_checked": destination_count,
        "api_results": validated_destination_count,
        "gemini_results": destination_count,
        "fallback_results": 0,
        "quality_check": f"{destination_count} destinations were generated from the profile and {validated_destination_count} were validated through Geoapify.",
        "passed_basic_check": destination_count == 10 and validated_destination_count == 10
    })

except Exception as e:
    recommendation_quality_rows.append({
        "recommendation_area": "Destination recommendations",
        "items_checked": 0,
        "api_results": 0,
        "gemini_results": 0,
        "fallback_results": 0,
        "quality_check": f"Could not audit destination recommendations: {e}",
        "passed_basic_check": False
    })


# Hotel recommendation quality
try:
    hotel_count = safe_len(hotel_options)

    local_hotels = [
        hotel for hotel in hotel_options
        if is_likely_local_to_selected_destination(hotel, selected_destination)
    ]

    recommendation_quality_rows.append({
        "recommendation_area": "Hotel recommendations",
        "items_checked": hotel_count,
        "api_results": hotel_count,
        "gemini_results": 0,
        "fallback_results": 0,
        "quality_check": f"{len(local_hotels)} of {hotel_count} Foursquare hotel results appear local to the selected destination.",
        "passed_basic_check": hotel_count > 0 and len(local_hotels) == hotel_count
    })

except Exception as e:
    recommendation_quality_rows.append({
        "recommendation_area": "Hotel recommendations",
        "items_checked": 0,
        "api_results": 0,
        "gemini_results": 0,
        "fallback_results": 0,
        "quality_check": f"Could not audit hotel recommendations: {e}",
        "passed_basic_check": False
    })


# Activity recommendation quality
try:
    activity_rows = []
    total_activity_items = 0
    api_activity_items = 0
    gemini_activity_items = 0
    fallback_activity_items = 0
    required_project_items = 0
    local_activity_items = 0

    for category, places in results_by_category.items():
        category_total = safe_len(places)
        category_api = 0
        category_gemini = 0
        category_fallback = 0
        category_required = 0
        category_local = 0

        for place in places:
            result_source = str(place.get("result_source") or place.get("source") or "").lower()

            if "foursquare" in result_source:
                category_api += 1

            if "gemini" in result_source:
                category_gemini += 1

            if "fallback" in result_source:
                category_fallback += 1

            if "required project handoff" in result_source:
                category_required += 1

            if is_likely_local_to_selected_destination(place, selected_destination):
                category_local += 1

        total_activity_items += category_total
        api_activity_items += category_api
        gemini_activity_items += category_gemini
        fallback_activity_items += category_fallback
        required_project_items += category_required
        local_activity_items += category_local

        activity_rows.append({
            "activity_category": category,
            "total_results": category_total,
            "foursquare_api_results": category_api,
            "gemini_fallback_results": category_gemini,
            "required_project_results": category_required,
            "local_results": category_local,
            "passed_basic_check": category_total == 5 and category_local == category_total
        })

    activity_quality_df = pd.DataFrame(activity_rows)

    recommendation_quality_rows.append({
        "recommendation_area": "Specific activity recommendations",
        "items_checked": total_activity_items,
        "api_results": api_activity_items,
        "gemini_results": gemini_activity_items,
        "fallback_results": fallback_activity_items,
        "quality_check": f"{local_activity_items} of {total_activity_items} Foursquare/Gemini activity results appear local to the selected destination.",
        "passed_basic_check": total_activity_items > 0 and local_activity_items == total_activity_items
    })

except Exception as e:
    activity_quality_df = pd.DataFrame()

    recommendation_quality_rows.append({
        "recommendation_area": "Specific activity recommendations",
        "items_checked": 0,
        "api_results": 0,
        "gemini_results": 0,
        "fallback_results": 0,
        "quality_check": f"Could not audit activity recommendations: {e}",
        "passed_basic_check": False
    })


# Build recommendation quality dataframe
recommendation_quality_df = pd.DataFrame(recommendation_quality_rows)

# Calculate recommendation quality pass rate
recommendation_quality_rate = recommendation_quality_df["passed_basic_check"].mean()

# Print recommendation quality audit
print("\n" + "=" * 80)
print("RECOMMENDATION QUALITY AUDIT")
print("=" * 80)
print("This is not ground-truth factual accuracy. It checks source type, coverage, and local relevance.")
display(recommendation_quality_df)

print("Recommendation quality pass rate:", round(recommendation_quality_rate, 3))

# Print activity result source breakdown
if not activity_quality_df.empty:
    print("\n" + "=" * 80)
    print("ACTIVITY RESULT SOURCE BREAKDOWN")
    print("=" * 80)
    display(activity_quality_df)


# Travel-specific edge-case tests
edge_case_rows = []

# Edge case 1: destination generation should return 10 validated options
print("\n" + "=" * 80)
print("TRAVEL EDGE-CASE TEST")
print("Test case: Profile-based destination generation")
print("Expected behavior: Return 10 Geoapify-validated destination options.")
print("Tool being tested: recommend_destinations")

try:
    destination_edge_json = recommend_destinations(
        profile_json=json.dumps(profile_without_destination),
        required_destination="Waikiki, Honolulu, Hawaii",
        total_options=10
    )

    destination_edge_data = json.loads(destination_edge_json)
    destination_edge_count = len(destination_edge_data.get("destinations", []))

    print("Actual behavior:", f"Returned {destination_edge_count} validated destinations.")
    print("Handled gracefully:", destination_edge_count == 10)

    edge_case_rows.append({
        "test_case": "Profile-based destination generation",
        "expected_behavior": "Return 10 Geoapify-validated destination options.",
        "actual_behavior": f"Returned {destination_edge_count} validated destinations.",
        "handled_gracefully": destination_edge_count == 10,
        "tool_used": "recommend_destinations"
    })

except Exception as e:
    print("Actual behavior:", f"Crashed: {e}")
    print("Handled gracefully:", False)

    edge_case_rows.append({
        "test_case": "Profile-based destination generation",
        "expected_behavior": "Return 10 Geoapify-validated destination options.",
        "actual_behavior": f"Crashed: {e}",
        "handled_gracefully": False,
        "tool_used": "recommend_destinations"
    })


# Edge case 2: weak API activity category should still get 5 fallback-filled results
print("\n" + "=" * 80)
print("TRAVEL EDGE-CASE TEST")
print("Test case: Weak API activity category")
print("Expected behavior: Return 5 results using Gemini fallback if Foursquare is limited.")
print("Tool being tested: search_activity_options")

try:
    weak_activity_test_json = search_activity_options(
        destination=selected_destination,
        selected_activity_categories_json=json.dumps(["private cabana"]),
        limit_per_category=5
    )

    weak_activity_test = json.loads(weak_activity_test_json)
    weak_places = weak_activity_test["results_by_category"].get("private cabana", [])

    print("Actual behavior:", f"Returned {len(weak_places)} results.")
    print("Handled gracefully:", len(weak_places) == 5)

    edge_case_rows.append({
        "test_case": "Weak API activity category",
        "expected_behavior": "Return 5 results using Gemini fallback if Foursquare is limited.",
        "actual_behavior": f"Returned {len(weak_places)} results.",
        "handled_gracefully": len(weak_places) == 5,
        "tool_used": "search_activity_options"
    })

except Exception as e:
    print("Actual behavior:", f"Crashed: {e}")
    print("Handled gracefully:", False)

    edge_case_rows.append({
        "test_case": "Weak API activity category",
        "expected_behavior": "Return 5 results using Gemini fallback if Foursquare is limited.",
        "actual_behavior": f"Crashed: {e}",
        "handled_gracefully": False,
        "tool_used": "search_activity_options"
    })


# Edge case 3: hotel category alias should work
print("\n" + "=" * 80)
print("TRAVEL EDGE-CASE TEST")
print("Test case: Hotel category alias")
print("Expected behavior: Treat accommodation.hotel as a valid hotel category.")
print("Tool being tested: search_places")

try:
    alias_hotel_test_json = search_places(
        destination=selected_destination,
        category="accommodation.hotel",
        limit=5
    )

    alias_hotel_test = json.loads(alias_hotel_test_json)
    alias_hotels = alias_hotel_test.get("places", [])

    print("Actual behavior:", f"Returned {len(alias_hotels)} hotel results.")
    print("Handled gracefully:", len(alias_hotels) > 0)

    edge_case_rows.append({
        "test_case": "Hotel category alias",
        "expected_behavior": "Treat accommodation.hotel as a valid hotel category.",
        "actual_behavior": f"Returned {len(alias_hotels)} hotel results.",
        "handled_gracefully": len(alias_hotels) > 0,
        "tool_used": "search_places"
    })

except Exception as e:
    print("Actual behavior:", f"Crashed: {e}")
    print("Handled gracefully:", False)

    edge_case_rows.append({
        "test_case": "Hotel category alias",
        "expected_behavior": "Treat accommodation.hotel as a valid hotel category.",
        "actual_behavior": f"Crashed: {e}",
        "handled_gracefully": False,
        "tool_used": "search_places"
    })


# Edge case 4: invalid itinerary day should be rejected
print("\n" + "=" * 80)
print("TRAVEL EDGE-CASE TEST")
print("Test case: Invalid itinerary day")
print("Expected behavior: Reject a day outside the trip length.")
print("Tool being tested: build_itinerary")

try:
    invalid_assignment = [{
        "activity_index": 0,
        "activity_name": selected_specific_activities[0].get("name"),
        "day": f"Day {int(user_profile['trip_length_days']) + 1}",
        "time_slot": "Morning"
    }]

    build_itinerary(
        profile_json=json.dumps(user_profile),
        selected_destination=selected_destination,
        selected_hotel_json=json.dumps(selected_hotel),
        selected_activities_json=json.dumps(selected_specific_activities),
        assignments_json=json.dumps(invalid_assignment)
    )

    print("Actual behavior:", "Invalid day was accepted.")
    print("Handled gracefully:", False)

    edge_case_rows.append({
        "test_case": "Invalid itinerary day",
        "expected_behavior": "Reject a day outside the trip length.",
        "actual_behavior": "Invalid day was accepted.",
        "handled_gracefully": False,
        "tool_used": "build_itinerary"
    })

except Exception as e:
    print("Actual behavior:", f"Rejected invalid day: {e}")
    print("Handled gracefully:", True)

    edge_case_rows.append({
        "test_case": "Invalid itinerary day",
        "expected_behavior": "Reject a day outside the trip length.",
        "actual_behavior": f"Rejected invalid day: {e}",
        "handled_gracefully": True,
        "tool_used": "build_itinerary"
    })


# Edge case 5: invalid itinerary time should be rejected
print("\n" + "=" * 80)
print("TRAVEL EDGE-CASE TEST")
print("Test case: Invalid itinerary time slot")
print("Expected behavior: Reject a time slot outside Morning/Afternoon/Evening.")
print("Tool being tested: build_itinerary")

try:
    invalid_time_assignment = [{
        "activity_index": 0,
        "activity_name": selected_specific_activities[0].get("name"),
        "day": "Day 1",
        "time_slot": "Brunch"
    }]

    build_itinerary(
        profile_json=json.dumps(user_profile),
        selected_destination=selected_destination,
        selected_hotel_json=json.dumps(selected_hotel),
        selected_activities_json=json.dumps(selected_specific_activities),
        assignments_json=json.dumps(invalid_time_assignment)
    )

    print("Actual behavior:", "Invalid time slot was accepted.")
    print("Handled gracefully:", False)

    edge_case_rows.append({
        "test_case": "Invalid itinerary time slot",
        "expected_behavior": "Reject a time slot outside Morning/Afternoon/Evening.",
        "actual_behavior": "Invalid time slot was accepted.",
        "handled_gracefully": False,
        "tool_used": "build_itinerary"
    })

except Exception as e:
    print("Actual behavior:", f"Rejected invalid time slot: {e}")
    print("Handled gracefully:", True)

    edge_case_rows.append({
        "test_case": "Invalid itinerary time slot",
        "expected_behavior": "Reject a time slot outside Morning/Afternoon/Evening.",
        "actual_behavior": f"Rejected invalid time slot: {e}",
        "handled_gracefully": True,
        "tool_used": "build_itinerary"
    })


# Build edge-case dataframe
edge_case_df = pd.DataFrame(edge_case_rows)

# Calculate edge-case handling rate
edge_case_handling_rate = edge_case_df["handled_gracefully"].mean()

# Print edge-case table
print("\n" + "=" * 80)
print("TRAVEL EDGE-CASE HANDLING TABLE")
print("=" * 80)
display(edge_case_df)

print("Travel edge-case handling rate:", round(edge_case_handling_rate, 3))


# Final audit summary
print("\n" + "=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)
print(f"Workflow tool accuracy: {round(workflow_tool_accuracy, 3)}")
print(f"Agent routing accuracy: {round(agent_routing_accuracy, 3)}")
print(f"Agent routing full success rate: {round(agent_routing_success_rate, 3)}")
print(f"Recommendation quality pass rate: {round(recommendation_quality_rate, 3)}")
print(f"Travel edge-case handling rate: {round(edge_case_handling_rate, 3)}")

print("\nSummary:")
print("- Destination options are generated from the user profile and validated with Geoapify.")
print("- Foursquare is used for hotel and activity listings.")
print("- Waikiki is intentionally required as one destination option for the later CV handoff.")
print("- Workflow tool accuracy checks direct app workflow tool calls.")
print("- Agent routing accuracy checks whether Gemini chooses the expected tool without being told the tool name.")
print("- Recommendation quality checks source type, coverage, and local relevance, not ground-truth travel quality.")
print("- Edge cases test travel-specific failures.")
print("- The itinerary builder is counted as a tool because Cell 8 calls build_itinerary().")

WORKFLOW TOOL AUDIT
This table checks whether each major workflow step used the expected workflow tool.


,workflow_step,expected_tool,actual_tool,tool_type,notes,correct
0,Profile-based destination recommendations,recommend_destinations,recommend_destinations,Direct Python tool call,Gemini generated destination options from the ...,True
1,Budget-fit destination recommendations,estimate_budget_fit,estimate_budget_fit,Direct Python tool call,Gemini estimated budget fit because Geoapify d...,True
2,User profile validation,create_user_profile,create_user_profile,Direct Python tool call,Validated required profile fields and saved th...,True
3,Hotel options for selected destination,search_places,search_places,Direct Python tool call,Used Foursquare place listings to retrieve hot...,True
4,Specific activity options from selected catego...,search_activity_options,search_activity_options,Direct Python tool call,Used Foursquare first and Gemini fallback only...,True
5,Final itinerary construction,build_itinerary,build_itinerary,Direct Python tool call,Built the final itinerary from selected activi...,True


Workflow tool accuracy: 1.0

AGENT ROUTING TEST
Question: I need 10 destination ideas for this travel profile. Include Waikiki as one option for a later beach crowd analysis: {"origin": "Oklahoma City", "age": "38", "traveling_with_children": "No", "number_of_travelers": 2, "trip_length_days": 5, "total_trip_budget": 6000.0, "destination_types": ["beach", "wellness/spa", "physical/adventure", "food/culinary", "romantic/couples", "luxury/resort"], "hotel_budget_tier": "$$$", "hotel_style": "resort", "pace": "balanced", "crowd_tolerance": "medium"}
[TOOL CALL] recommend_destinations(profile_json='{"origin": "Oklahoma City", "age": "38", "traveling_with_children": "No", "number_of_travelers": 2, "trip_length_days": 5, "total_trip_budget": 6000.0, "destination_types": ["beach", "wellness/spa", "physical/adventure", "food/culinary", "romantic/couples", "luxury/resort"], "hotel_budget_tier": "$$$", "hotel_style": "resort", "pace": "balanced", "crowd_tolerance": "medium"}', total_options=10, 

Expected tool: search_activity_options
Actual tool: search_activity_options
Correct tool: True
Usable result: False

AGENT ROUTING TEST
Question: Write one short tagline for a travel-planning app.
Expected tool: none
Actual tool: none
Correct tool: True
Usable result: True

AGENT ROUTING ACCURACY TABLE
This table checks whether Gemini chose the expected tool without being told the tool name.


,question,expected_tool,actual_tool,correct_tool,usable_result,fully_successful
0,I need 10 destination ideas for this travel pr...,recommend_destinations,recommend_destinations,True,True,True
1,A hotel costs 325 dollars per night for 3 nigh...,calculator,calculator,True,True,True
2,What time is it right now?,get_datetime,get_datetime,True,True,True
3,"Find hotel options near Waikīkī, Honolulu, HI,...",search_places,search_places,True,True,True
4,What specific activity options can I do near W...,search_activity_options,search_activity_options,True,False,False
5,Write one short tagline for a travel-planning ...,none,none,True,True,True


...

Recommendation quality pass rate: 0.667

ACTIVITY RESULT SOURCE BREAKDOWN


,activity_category,total_results,foursquare_api_results,gemini_fallback_results,required_project_results,local_results,passed_basic_check
0,beach time,5,4,0,1,5,True
1,spa treatment,5,5,0,0,5,True
2,yoga class,5,3,2,0,5,True
3,healthy dining,5,5,0,0,5,True
4,hiking,5,5,0,0,3,False
5,fine dining,5,5,0,0,5,True
6,private cabana,5,0,5,0,5,True



TRAVEL EDGE-CASE TEST
Test case: Profile-based destination generation
Expected behavior: Return 10 Geoapify-validated destination options.
Tool being tested: recommend_destinations
Actual behavior: Returned 10 validated destinations.
Handled gracefully: True

TRAVEL EDGE-CASE TEST
Test case: Weak API activity category
Expected behavior: Return 5 results using Gemini fallback if Foursquare is limited.
Tool being tested: search_activity_options
Actual behavior: Returned 5 results.
Handled gracefully: True

TRAVEL EDGE-CASE TEST
Test case: Hotel category alias
Expected behavior: Treat accommodation.hotel as a valid hotel category.
Tool being tested: search_places
Actual behavior: Returned 5 hotel results.
Handled gracefully: True

TRAVEL EDGE-CASE TEST
Test case: Invalid itinerary day
Expected behavior: Reject a day outside the trip length.
Tool being tested: build_itinerary
Actual behavior: Rejected invalid day: Day 6 is outside the trip length.
Handled gracefully: True

TRAVEL EDGE-CAS

,test_case,expected_behavior,actual_behavior,handled_gracefully,tool_used
0,Profile-based destination generation,Return 10 Geoapify-validated destination options.,Returned 10 validated destinations.,True,recommend_destinations
1,Weak API activity category,Return 5 results using Gemini fallback if Four...,Returned 5 results.,True,search_activity_options
2,Hotel category alias,Treat accommodation.hotel as a valid hotel cat...,Returned 5 hotel results.,True,search_places
3,Invalid itinerary day,Reject a day outside the trip length.,Rejected invalid day: Day 6 is outside the tri...,True,build_itinerary
4,Invalid itinerary time slot,Reject a time slot outside Morning/Afternoon/E...,Rejected invalid time slot: Time slot must be ...,True,build_itinerary


Travel edge-case handling rate: 1.0

AUDIT SUMMARY
Workflow tool accuracy: 1.0
Agent routing accuracy: 1.0
Agent routing full success rate: 0.833
Recommendation quality pass rate: 0.667
Travel edge-case handling rate: 1.0

Summary:
- Destination options are generated from the user profile and validated with Geoapify.
- Foursquare is used for hotel and activity listings.
- Waikiki is intentionally required as one destination option for the later CV handoff.
- Workflow tool accuracy checks direct app workflow tool calls.
- Agent routing accuracy checks whether Gemini chooses the expected tool without being told the tool name.
- Recommendation quality checks source type, coverage, and local relevance, not ground-truth travel quality.
- Edge cases test travel-specific failures.
- The itinerary builder is counted as a tool because Cell 8 calls build_itinerary().


###Cell 9 Output
### Cell 9 Summary
Workflow used the expected tools for every step, giving it a workflow tool accuracy of 1.000. Gemini also chose the correct tool for every routing test. One activity-search test failed because Gemini returned a high-demand error, so the full routing success rate dropped to 0.833. Destination and hotel recommendations passed the basic quality checks, while the activity recommendations were weaker because 2 hiking results did not appear local to Waikiki, but they are local. All edge cases were handled correctly.

### Cell 10: Final Itinerary Display

| Step | What | Why |
|---|---|---|
| 1 | The final itinerary JSON is loaded. | Prepares the selected trip plan for display. |
| 2 | The notebook prints the destination and selected hotel. | Shows the main trip choices at the top. |
| 3 | Each trip day is printed with Morning, Afternoon, and Evening sections. | Organizes the schedule into a readable daily itinerary. |
| 4 | Scheduled activities show name, category, address, and source. | Keeps each itinerary item traceable. |
| 5 | Empty time slots are labeled open. | Shows where the itinerary still has room. |

In [16]:
# Cell 10: Print Final Itinerary - display destination, hotel, and day-by-day itinerary

# Load final itinerary from Cell 8
final_itinerary = json.loads(final_itinerary_json)

# Print formatted itinerary
print("=" * 80)
print("FINAL ITINERARY")
print("=" * 80)

print("\nDESTINATION")
print(final_itinerary["destination"])

print("\nHOTEL")
print(final_itinerary["hotel"].get("name"))
print(final_itinerary["hotel"].get("address"))

print("\nDAILY PLAN")

# Print each day in order
for day_num in range(1, int(final_itinerary["trip_length_days"]) + 1):
    day = f"Day {day_num}"
    slots = final_itinerary["daily_itinerary"].get(day, {})

    print("\n" + "-" * 80)
    print(day.upper())

    for slot in ["Morning", "Afternoon", "Evening"]:
        print(f"\n{slot}:")

        activities = slots.get(slot, [])

        if not activities:
            print("- Open")
        else:
            for activity in activities:
                print(f"- {activity.get('name')}")
                print(f"  Category: {activity.get('matched_activity_category') or activity.get('matched_category')}")
                print(f"  Address: {activity.get('address')}")
                print(f"  Source: {activity.get('result_source') or activity.get('source')}")

print("\n" + "=" * 80)
print("END OF ITINERARY")
print("=" * 80)

FINAL ITINERARY

DESTINATION
Waikīkī, Honolulu, HI, United States of America

HOTEL
Park Shore Waikiki Hotel
2586 Kalakaua Ave (Kapahulu Ave.), Honolulu, HI 96815

DAILY PLAN

--------------------------------------------------------------------------------
DAY 1

Morning:
- Open

Afternoon:
- Open

Evening:
- Open

--------------------------------------------------------------------------------
DAY 2

Morning:
- Open

Afternoon:
- Waikiki Beach
  Category: beach time
  Address: Waikiki, Honolulu, Hawaii
  Source: Required project handoff option

Evening:
- Open

--------------------------------------------------------------------------------
DAY 3

Morning:
- Open Space Yoga (Diamond Head)
  Category: yoga class
  Address: 3106 Monsarrat Avenue (at Kanaina Ave), Honolulu, HI 96815
  Source: Foursquare API

Afternoon:
- Park Restaurant Waikiki
  Category: healthy dining
  Address: 2885 Kalakaua Ave, Honolulu, HI 96815
  Source: Foursquare API

Evening:
- Open

--------------------------

### Cell 5 Markdown: Multi-Turn Tool Use

**Question:** Did the model correctly decide when to use tools and when to answer directly? Were there any cases where it made the wrong decision? What would you change about the tool descriptions to improve its routing?

**Answer:** The model chose the correct tool in each routing test and answered directly when no tool was needed. The tool descriptions worked because they separated destination, hotel, activity, math, and date/time tasks.

### Cell 6 Markdown: Tool Routing Accuracy

**Question:** What was your routing accuracy? Were the errors predictable? For the questions where the model made the wrong routing decision, could you rewrite the tool description to fix it without breaking something else?

**Answer:** The routing accuracy was 1.000, meaning Gemini chose the expected tool in every routing test. One full-success score dropped because a temporary Gemini 503 error made one activity-search result unusable, not because the wrong tool was chosen.

### Cell 7 Markdown: Edge-Case Handling

**Question:** How did the model handle tool errors? Did it acknowledge the failure, or did it try to cover it up? What does this tell you about the model's relationship to its tools — does it trust them unconditionally?

**Answer:** The travel edge cases were handled correctly. The itinerary tool rejected invalid days and time slots, and weak activity categories still returned results using Gemini fallback.